# Aula 5, Orquestração com LangGraph

Notebook executável que acompanha a aula [05-orquestracao-langgraph.md](../../lessons/modulo-10-agentes/05-orquestracao-langgraph.md).

## O que vamos fazer aqui

Vamos modelar o agente como um grafo de estados: nós que transformam um estado compartilhado,
ligados por arestas. Implementamos do zero e mostramos a versão com LangGraph como opção. Python
puro.

## Agente como máquina de estados

Cada nó (decidir, executar, responder) recebe e devolve o estado. A orquestração liga os nós na
ordem certa.

In [ ]:
import re
import ast
import operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow}


def calcular(expr):
    def ev(no):
        if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
            return no.value
        if isinstance(no, ast.BinOp) and type(no.op) in OPS:
            return OPS[type(no.op)](ev(no.left), ev(no.right))
        raise ValueError("expressão não permitida")
    return ev(ast.parse(str(expr), mode="eval").body)


def buscar(consulta):
    base = {"derivada": "A derivada mede a taxa de variação de uma função."}
    for chave, texto in base.items():
        if chave in consulta.lower():
            return texto
    return "Não encontrei no material."


def no_decidir(estado):
    p = estado["pergunta"]
    estado["acao"] = "calcular" if (re.search(r"\d", p) and re.search(r"[+\-*/]", p)) else "buscar"
    return estado


def no_executar(estado):
    if estado["acao"] == "calcular":
        expr = "".join(re.findall(r"\d+\.?\d*|[+\-*/()]", estado["pergunta"]))
        estado["observacao"] = calcular(expr)
    else:
        estado["observacao"] = buscar(estado["pergunta"])
    return estado


def no_responder(estado):
    estado["resposta"] = f"Resultado: {estado['observacao']}"
    return estado


def executar_grafo(pergunta):
    estado = {"pergunta": pergunta}
    for no in (no_decidir, no_executar, no_responder):
        estado = no(estado)
    return estado


for p in ["quanto é 28*3/4 ?", "o que é a derivada?"]:
    print(p, "->", executar_grafo(p)["resposta"])

O agente percorre o grafo: decide a ação, executa a ferramenta e responde. Cada nó é uma
função pequena que transforma o estado, e a orquestração liga os nós. Para agentes com muitos
caminhos e ciclos, é essa estrutura que o LangGraph organiza.

O projeto que fecha o módulo é o agente tutor completo, em `projects/m10-tutor-agent/`, reunindo
ferramentas, planejamento, memória e orquestração.